In [0]:
import requests

url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": 21.034408,
    "longitude": 79.032069,

    "hourly": (
        "temperature_2m,"
        "relative_humidity_2m,"
        "dew_point_2m,"
        "apparent_temperature,"
        "precipitation_probability,"
        "precipitation,"
        "rain,"
        "showers,"
        "weather_code,"
        "cloud_cover,"
        "visibility,"
        "pressure_msl,"
        "surface_pressure,"
        "wind_speed_10m,"
        "wind_direction_10m,"
        "wind_gusts_10m"
    ),

    "timezone": "Asia/Kolkata",
    "forecast_days": 7
}

response = requests.get(
    url,
    params=params
)

response.raise_for_status()

data = response.json()["hourly"]

hourly_rows = list(
    zip(
        data["time"],
        data["temperature_2m"],
        data["relative_humidity_2m"],
        data["dew_point_2m"],
        data["apparent_temperature"],
        data["precipitation_probability"],
        data["precipitation"],
        data["rain"],
        data["showers"],
        data["weather_code"],
        data["cloud_cover"],
        data["visibility"],
        data["pressure_msl"],
        data["surface_pressure"],
        data["wind_speed_10m"],
        data["wind_direction_10m"],
        data["wind_gusts_10m"]
    )
)

schema = """
time STRING,
temperature_2m DOUBLE,
relative_humidity_2m LONG,
dew_point_2m DOUBLE,
apparent_temperature DOUBLE,
precipitation_probability LONG,
precipitation DOUBLE,
rain DOUBLE,
showers DOUBLE,
weather_code LONG,
cloud_cover LONG,
visibility DOUBLE,
pressure_msl DOUBLE,
surface_pressure DOUBLE,
wind_speed_10m DOUBLE,
wind_direction_10m LONG,
wind_gusts_10m DOUBLE
"""

df = spark.createDataFrame(
    hourly_rows,
    schema=schema
)




# df.write \
#     .format("delta") \
#     .mode("overwrite") \
#     .saveAsTable("workspace.weather_bronze.weather_hourly_new")

In [0]:
%sql 
delete from workspace.weather_bronze.weather_hourly_new
where date(time) >= current_date() 
;




In [0]:
df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.weather_bronze.weather_hourly_new")



In [0]:
# %sql
# CREATE TABLE IF NOT EXISTS workspace.weather_bronze.weather_hourly_new (
#   time STRING,
#   temperature_2m DOUBLE,
#   relative_humidity_2m LONG,
#   dew_point_2m DOUBLE,
#   apparent_temperature DOUBLE,
#   precipitation_probability LONG,
#   precipitation DOUBLE,
#   rain DOUBLE,
#   showers DOUBLE,
#   weather_code LONG,
#   cloud_cover LONG,
#   visibility DOUBLE,
#   pressure_msl DOUBLE,
#   surface_pressure DOUBLE,
#   wind_speed_10m DOUBLE,
#   wind_direction_10m LONG,
#   wind_gusts_10m DOUBLE,
#   at_created TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
# );

In [0]:
%sql
-- Enable column defaults feature
ALTER TABLE workspace.weather_bronze.weather_hourly_new 
SET TBLPROPERTIES('delta.feature.allowColumnDefaults' = 'supported');

-- Set default value for future inserts
ALTER TABLE workspace.weather_bronze.weather_hourly_new 
ALTER COLUMN at_created SET DEFAULT CURRENT_TIMESTAMP();

-- Backfill existing rows
UPDATE workspace.weather_bronze.weather_hourly_new 
SET at_created = CURRENT_TIMESTAMP() 
WHERE at_created IS NULL;

In [0]:
%sql
select * from workspace.weather_bronze.weather_hourly_new ;